In [ ]:
# m4.3 -- rolling averages with .rolling()
# .rolling() calculates the average over a sliding window of rows
# instead of collapsing the entire Series into a single number like .agg()
# .rolling() calculates a new value for each row based on N rows before it

# example with window size of 3 (not N-1 rows will produce NaN in this example)
# window size of 3 means .rolling(3)
# to calculate the mean for Row N
# Pandas will need rows N-2, N-1, and N --> this is the first 3 rows

# below example of NaN's that appear untilt the window contains 3 values for .rolling(3)
# eacch rows rolling mean looks backward - it will include the current row and the N-1 rows before it
# sample code: df["daily_demand"].rolling(3).mean()

#       value window contains        rolling mean (window = N = 3)   
# row 1: 100: [100]               --> NaN (not enough rows yet)       (1-1 - 0)
# row 2:  85: [100, 85]           --> NaN (not enough rows yet        (2-1 = 1)
# row 3: 120: [100, 85, 120]      --> mean = 100.17                   (3-1 = 2)
# row 4:  95: [85, 120, 95]       --> mean = 100                      (4-1 = 3)
# row 5: 110: [120, 95, 110]      --> mean = 108.3                    (5-1 = 4)

In [ ]:
# import libraries
import pandas as pd
import numpy as np

# set the seed
# the seed is the initial value that determines the sequence of numbers the pseudo random number generator produces
# by setting the seed at 42 or etc you fix the seed 
# esnuring your code will produce the exact same sequence of random numbers every time it is run 
np.random.seed(42)

# create the dataframe
dates = pd.date_range(start="2024-01-01", end="2024-03-31", freq="D")
df = pd.DataFrame ({
    "sku_id" : "SKU-001",
    "daily_demand" : np.random.randint(low=50, high=150, size=len(dates)),
    },
    index=dates 
)
print(df.index)
print(df.head(10))

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-12',
               '2024-01-13', '2024-01-14', '2024-01-15', '2024-01-16',
               '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20',
               '2024-01-21', '2024-01-22', '2024-01-23', '2024-01-24',
               '2024-01-25', '2024-01-26', '2024-01-27', '2024-01-28',
               '2024-01-29', '2024-01-30', '2024-01-31', '2024-02-01',
               '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05',
               '2024-02-06', '2024-02-07', '2024-02-08', '2024-02-09',
               '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13',
               '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-17',
               '2024-02-18', '2024-02-19', '2024-02-20', '2024-02-21',
               '2024-02-22', '2024-02-23', '2024-02-24', '2024-02-25',
      

In [ ]:
# exercise 2
# show the first 30 rows of daily_demand, rolling_7, rolling_!4, and rolling_28 side-by-side
# notice N-1 rows of NaN (7-1=6, 14-1=13, and 28-1=27 NaN's per Series)
df_roll = (
    df
    .assign(
        rolling_7 = lambda x: x["daily_demand"].rolling(window=7).mean(),
        rolling_14 = lambda x: x["daily_demand"].rolling(window=14).mean(),
        rolling_28 = lambda x: x["daily_demand"].rolling(window=28).mean()
    )
)

print(df_roll[["daily_demand", "rolling_7", "rolling_14", "rolling_28"]].head(30))

In [5]:
# exercise 3

# Filter rows where rolling_7 is greater than 110
df_gt = df_roll[df_roll["rolling_7"] > 110]
print(df_gt)

# Filter rows where rolling_28 is less than 90
df_lt = df_roll[df_roll["rolling_28"] <90]
print(df_lt)

             sku_id  daily_demand   rolling_7  rolling_14  rolling_28
2024-01-08  SKU-001           136  110.714286         NaN         NaN
2024-01-10  SKU-001           124  116.714286         NaN         NaN
2024-01-11  SKU-001           137  119.000000         NaN         NaN
2024-01-12  SKU-001           149  124.571429         NaN         NaN
2024-01-13  SKU-001            73  125.000000         NaN         NaN
2024-01-14  SKU-001            52  113.571429  109.642857         NaN
2024-02-01  SKU-001           108  112.428571   98.428571  101.642857
2024-02-03  SKU-001           141  112.428571  103.142857  103.500000
2024-02-04  SKU-001           109  117.857143  107.285714  102.678571
2024-02-05  SKU-001           129  116.571429  108.428571  102.428571
2024-02-06  SKU-001            64  111.714286  105.214286  100.285714
2024-03-03  SKU-001           120  113.571429   97.928571   96.785714
Empty DataFrame
Columns: [sku_id, daily_demand, rolling_7, rolling_14, rolling_28]
Index: 

In [ ]:
# exercise 4
# rolling standard deviation

# why is the standard deviation useful for safety stock calculations
# the standard deviation tells you how far the actual demand is deviating from the rolling average (in this case)
# a higher deviation requires more safety stock
# example: 2024-01-07 has daily demand of 132, rolling(7) = 105.71, and rolling_std_7 of 29.7
# so demand varies by about +-30 units around 105.71
# when the rolling_std_7 is lower, say 22.2, thtis suggests demand was more stable as there is less variability around the rolling(7) 
# you are measuring the unpredictability of demand within the rolling window

df_roll = (
    df_roll
    .assign(
        rolling_std_7 = lambda x: x["daily_demand"].rolling(7).std().round(1)
    )
)

print(df_roll[["daily_demand", "rolling_7", "rolling_std_7"]].head(n=30))

            daily_demand   rolling_7  rolling_std_7
2024-01-01           101         NaN            NaN
2024-01-02           142         NaN            NaN
2024-01-03            64         NaN            NaN
2024-01-04           121         NaN            NaN
2024-01-05           110         NaN            NaN
2024-01-06            70         NaN            NaN
2024-01-07           132  105.714286           29.7
2024-01-08           136  110.714286           31.7
2024-01-09           124  108.142857           29.4
2024-01-10           124  116.714286           22.2
2024-01-11           137  119.000000           23.5
2024-01-12           149  124.571429           25.5
2024-01-13            73  125.000000           24.5
2024-01-14            52  113.571429           36.4
2024-01-15            71  104.285714           38.0
2024-01-16           102  101.142857           37.0
2024-01-17            51   90.714286           39.7
2024-01-18           137   90.714286           39.7
2024-01-19  